# Week 6: Geometric Transforms — Lab Practice

**Topics:** Transformation Matrices, Affine Transforms, Homography & Document Scanning

This notebook accompanies the Week 6 lecture slides. We focus on two core skills:
1. **Building and applying geometric transformation matrices** — rotation, scaling, translation in homogeneous coordinates
2. **Homography** — computing a perspective transform from 4 point correspondences to "scan" a document

We also include a brief demo of the **Hough Transform** for line detection.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2

%matplotlib inline

### Environment setup

This notebook works both **locally** and on **Google Colab**.

In [ ]:
import os, urllib.request

# Detect Google Colab
IN_COLAB = "google.colab" in str(get_ipython()) if hasattr(__builtins__, "__IPYTHON__") else False

if IN_COLAB:
    REPO_URL = "https://raw.githubusercontent.com/HyeongminLEE/image-processing-tutorial/main"
    os.makedirs("images", exist_ok=True)
    for fname in ["parrots_square.jpg", "yangbanga.jpg", "calligraphy_hall.jpg"]:
        url = f"{REPO_URL}/images/{fname}"
        if not os.path.exists(f"images/{fname}"):
            urllib.request.urlretrieve(url, f"images/{fname}")
            print(f"Downloaded {fname}")
    IMG_DIR = "images/"
else:
    IMG_DIR = "../images/"

print(f"Running on: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Image directory: {IMG_DIR}")

### Display helpers

Utility functions used throughout this notebook. Each helper displays **one item** per call.

In [ ]:
def show_image(img, title=None, scale=4):
    """Display a single image.

    Grayscale (2-D) arrays automatically use a gray colormap with [0, 255].
    """
    fig, ax = plt.subplots(figsize=(scale, scale))
    if img.ndim == 2:
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    else:
        ax.imshow(img)
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_transform(pts_before, pts_after, title=None, scale=5):
    """Show original and transformed 2D points on the same axes.

    Points are connected as a closed polygon.
    Blue = original, Red = transformed.
    """
    fig, ax = plt.subplots(figsize=(scale, scale))
    for pts, color, marker, label in [
        (pts_before, "royalblue", "o", "Original"),
        (pts_after, "crimson", "s", "Transformed"),
    ]:
        closed = np.vstack([pts, pts[0:1]])
        ax.plot(closed[:, 0], closed[:, 1], f"-{marker}",
                color=color, markersize=8, label=label)
    all_pts = np.vstack([pts_before, pts_after])
    margin = 1.0
    ax.set_xlim(all_pts[:, 0].min() - margin, all_pts[:, 0].max() + margin)
    ax.set_ylim(all_pts[:, 1].min() - margin, all_pts[:, 1].max() + margin)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color="k", linewidth=0.5)
    ax.axvline(0, color="k", linewidth=0.5)
    ax.legend()
    if title:
        ax.set_title(title)
    plt.tight_layout()
    plt.show()

---
## 0. Hough Transform — Quick Demo

The **Hough Transform** detects parametric shapes (lines, circles) in edge maps using a **voting** scheme. Each edge pixel votes for all parameter combinations that could pass through it; peaks in the accumulator reveal the dominant shapes.

This is a brief demo — no exercises. See the lecture slides for the full theory (Cartesian vs polar parameterization, accumulator, etc.).

In [ ]:
# Load the same image used in the lecture slides (calligraphy hall sign)
img_hough = np.array(Image.open(IMG_DIR + "calligraphy_hall.jpg"))
gray_hough = np.array(Image.open(IMG_DIR + "calligraphy_hall.jpg").convert("L"))

edges = cv2.Canny(gray_hough, 50, 150)

show_image(gray_hough, title="Grayscale")
show_image(edges, title="Canny edges")

In [ ]:
# Standard Hough Transform (same code as lecture slides)
lines = cv2.HoughLines(edges, rho=1, theta=np.pi / 180, threshold=150)

# Convert (rho, theta) to endpoints and draw
result = img_hough.copy()
for line in lines:
    rho, theta = line[0]
    a, b = np.cos(theta), np.sin(theta)
    x0, y0 = a * rho, b * rho
    pt1 = (int(x0 + 1000 * (-b)), int(y0 + 1000 * (a)))
    pt2 = (int(x0 - 1000 * (-b)), int(y0 - 1000 * (a)))
    cv2.line(result, pt1, pt2, (255, 0, 0), 2)

print(f"Lines detected: {len(lines)}")
show_image(result, title=f"Hough lines ({len(lines)} detected)", scale=5)

The `threshold` parameter controls how many votes a line needs to be detected. Too low → many false detections; too high → misses real lines.

In [ ]:
# Compare different threshold values
def draw_hough_lines(img, edges, threshold):
    """Run HoughLines and draw results. Returns image and line count."""
    lines = cv2.HoughLines(edges, rho=1, theta=np.pi / 180, threshold=threshold)
    vis = img.copy()
    n = 0
    if lines is not None:
        n = len(lines)
        for line in lines:
            rho, theta = line[0]
            a, b = np.cos(theta), np.sin(theta)
            x0, y0 = a * rho, b * rho
            pt1 = (int(x0 + 1000 * (-b)), int(y0 + 1000 * (a)))
            pt2 = (int(x0 - 1000 * (-b)), int(y0 - 1000 * (a)))
            cv2.line(vis, pt1, pt2, (255, 0, 0), 2)
    return vis, n

vis_80, n_80 = draw_hough_lines(img_hough, edges, threshold=80)
vis_150, n_150 = draw_hough_lines(img_hough, edges, threshold=150)
vis_200, n_200 = draw_hough_lines(img_hough, edges, threshold=200)

show_image(vis_80, title=f"threshold=80  ({n_80} lines)", scale=4)
show_image(vis_150, title=f"threshold=150 ({n_150} lines)", scale=4)
show_image(vis_200, title=f"threshold=200 ({n_200} lines)", scale=4)

---
## 1. Geometric Transforms

A geometric transform maps every point $(x, y)$ to a new location $(x', y')$. We express this mapping as **matrix multiplication**:

$$\begin{pmatrix}x'\\y'\end{pmatrix} = \begin{pmatrix}a & b\\c & d\end{pmatrix}\begin{pmatrix}x\\y\end{pmatrix}$$

Different values of the matrix entries produce different transforms: rotation, scaling, shear, and more. We will build these matrices by hand and apply them to both **points** and **images**.

### Transforms on points

Let's start with a simple triangle and see how different matrices transform it.

In [ ]:
# Define a triangle as 3 points (each row is [x, y])
triangle = np.array([
    [1, 0],
    [0, 2],
    [-1, 0],
], dtype=np.float64)

print(f"Triangle vertices:\n{triangle}")

### Rotation

The rotation matrix for angle $\theta$:

$$R = \begin{pmatrix}\cos\theta & -\sin\theta\\\sin\theta & \cos\theta\end{pmatrix}$$

To transform all points at once: each point is a column, so we compute $R \cdot P^T$ and transpose back.

In [ ]:
theta = np.radians(45)

R = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

print(f"Rotation matrix (theta = 45 deg):\n{R}")

rotated = (R @ triangle.T).T

show_transform(triangle, rotated, title="Rotation by 45 deg")

### Scaling

A diagonal matrix scales $x$ and $y$ independently:

$$S = \begin{pmatrix}s_x & 0\\0 & s_y\end{pmatrix}$$

In [ ]:
S = np.array([
    [2.0, 0],
    [0,   0.5]
])

print(f"Scaling matrix:\n{S}")

scaled = (S @ triangle.T).T

show_transform(triangle, scaled, title="Scale: x * 2, y * 0.5")

### The problem: translation

Rotation and scaling are **matrix multiplication**. But translation is **addition**:

$$\begin{pmatrix}x'\\y'\end{pmatrix} = \begin{pmatrix}x + t_x\\y + t_y\end{pmatrix}$$

This breaks chaining — we cannot combine rotation and translation into a single $2 \times 2$ matrix. The solution: **homogeneous coordinates**.

### Homogeneous coordinates

Add a third coordinate (always 1) to every point:

$$\begin{pmatrix}x\\y\end{pmatrix} \longrightarrow \begin{pmatrix}x\\y\\1\end{pmatrix}$$

Now **all transforms** — including translation — become $3 \times 3$ matrix multiplication:

$$\begin{pmatrix}x'\\y'\\1\end{pmatrix} = \begin{pmatrix}1&0&t_x\\0&1&t_y\\0&0&1\end{pmatrix}\begin{pmatrix}x\\y\\1\end{pmatrix} = \begin{pmatrix}x + t_x\\y + t_y\\1\end{pmatrix}$$

In [ ]:
# Convert to homogeneous coordinates: [x, y] → [x, y, 1]
triangle_h = np.hstack([triangle, np.ones((3, 1))])

print(f"Homogeneous coordinates:\n{triangle_h}")

In [ ]:
# Translation matrix (3x3)
tx, ty = 3, 2

T = np.array([
    [1, 0, tx],
    [0, 1, ty],
    [0, 0, 1],
], dtype=np.float64)

print(f"Translation matrix (tx={tx}, ty={ty}):\n{T}")

translated_h = (T @ triangle_h.T).T
print(f"\nHomogeneous result:\n{translated_h}")

# Convert back to 2D: divide by w (3rd component), then take first 2
w_col = translated_h[:, 2:3]       # w = 1 for affine, but good practice
translated = translated_h[:, :2] / w_col

show_transform(triangle, translated, title=f"Translation by ({tx}, {ty})")

### Transforms on images

To warp an entire image, we use `cv2.warpAffine(img, M, (width, height))`.

- `M` is **$2 \times 3$** (the last row $[0, 0, 1]$ is implicit) — pass `M_3x3[:2]`
- OpenCV uses **inverse mapping** internally: no holes in the result

Let's see every transform type applied to a real image.

In [ ]:
img = np.array(Image.open(IMG_DIR + "parrots_square.jpg"))

print(f"Shape: {img.shape}, dtype: {img.dtype}")
show_image(img, title="Original", scale=5)

**Translation** — shift every pixel by $(t_x, t_y)$:

In [ ]:
h, w = img.shape[:2]

M_trans = np.array([
    [1, 0, 300],
    [0, 1, 200],
    [0, 0, 1]
], dtype=np.float64)

warped_trans = cv2.warpAffine(img, M_trans[:2], (w, h))
show_image(warped_trans, title="Translation (tx=300, ty=200)", scale=5)

**Rotation** — around the origin (top-left corner). Notice how most of the image moves out of the frame:

In [ ]:
theta = np.radians(15)

M_rot = np.array([
    [np.cos(theta), -np.sin(theta), 0],
    [np.sin(theta),  np.cos(theta), 0],
    [0,              0,             1]
], dtype=np.float64)

warped_rot = cv2.warpAffine(img, M_rot[:2], (w, h))
show_image(warped_rot, title="Rotation (theta=15 deg, around origin)", scale=5)

**Rigid** (3 DOF) — rotation + translation. We can write the matrix **directly**, or **compose** $T \cdot R$. Both give the same result:

In [ ]:
theta = np.radians(15)

# Direct: write the rigid matrix in one shot
M_rigid_direct = np.array([
    [np.cos(theta), -np.sin(theta), 200],
    [np.sin(theta),  np.cos(theta), 100],
    [0,              0,              1]
])

# Composed: multiply R and T separately
R = np.array([[np.cos(theta), -np.sin(theta), 0],
              [np.sin(theta),  np.cos(theta), 0],
              [0, 0, 1]])
T = np.array([[1, 0, 200],
              [0, 1, 100],
              [0, 0, 1]], dtype=np.float64)
M_rigid_composed = T @ R

print(f"Same matrix? {np.allclose(M_rigid_direct, M_rigid_composed)}")

warped_rigid = cv2.warpAffine(img, M_rigid_direct[:2], (w, h))
show_image(warped_rigid, title="Rigid (theta=15 deg, tx=200, ty=100)", scale=5)

**Similarity** (4 DOF) — scale + rotation + translation. Same idea — direct or composed:

In [ ]:
theta = np.radians(20)
s = 0.6

# Direct: s multiplied into the rotation block
M_sim_direct = np.array([
    [s * np.cos(theta), -s * np.sin(theta), 400],
    [s * np.sin(theta),  s * np.cos(theta), 300],
    [0,                  0,                  1]
])

# Composed: S @ R @ T
S = np.array([[s, 0, 0], [0, s, 0], [0, 0, 1]])
R = np.array([[np.cos(theta), -np.sin(theta), 0],
              [np.sin(theta),  np.cos(theta), 0],
              [0, 0, 1]])
T = np.array([[1, 0, 400], [0, 1, 300], [0, 0, 1]], dtype=np.float64)
M_sim_composed = T @ R @ S

print(f"Same matrix? {np.allclose(M_sim_direct, M_sim_composed)}")

warped_sim = cv2.warpAffine(img, M_sim_direct[:2], (w, h))
show_image(warped_sim, title="Similarity (s=0.6, theta=20 deg, tx=400, ty=300)", scale=5)

**Affine** (6 DOF) — the most general transform with last row $(0, 0, 1)$. The $2 \times 2$ block has no special structure ($a \neq d$, $b \neq -c$ in general), allowing shear, non-uniform scaling, etc. Parallel lines are still preserved.

In [ ]:
# General affine: scale + rotation + shear + translation all at once
M_affine_general = np.array([
    [0.8, 0.3, 200],
    [0.1, 0.7, 150],
    [0,   0,   1]
], dtype=np.float64)

warped_affine = cv2.warpAffine(img, M_affine_general[:2], (w, h))
show_image(warped_affine, title="General affine (no special structure in 2x2 block)", scale=5)

### Exercises

**Exercise 1.1:** Scale the parrots image (`img`) by 0.7 and rotate by 30 deg, both **around its center**.

The demos above transform around the **origin** (top-left corner), so the image flies off-screen. To transform around the **center**, you need to:
1. **Translate** the center to the origin: $T_1$ with $(t_x, t_y) = (-c_x, -c_y)$
2. **Scale** by $s = 0.7$: $S$
3. **Rotate** by $\theta = 30°$: $R$
4. **Translate** back: $T_2$ with $(t_x, t_y) = (c_x, c_y)$

Compose them: $M = T_2 \cdot R \cdot S \cdot T_1$, then apply with `cv2.warpAffine(img, M[:2], (w, h))`.

*Hint:* The image center is `(cx, cy) = (w // 2, h // 2)`. Build each as a $3 \times 3$ matrix and multiply with `@`.

In [ ]:
# Input
show_image(img, title="img", scale=5)

# YOUR CODE HERE
# cx, cy = w // 2, h // 2
# theta = np.radians(30)
# s = 0.7
# 1. T1 = 3x3 translation matrix with tx=-cx, ty=-cy
# 2. S  = 3x3 scale matrix (s along both axes)
# 3. R  = 3x3 rotation matrix for theta
# 4. T2 = 3x3 translation matrix with tx=cx, ty=cy
# 5. M  = T2 @ R @ S @ T1
# 6. warped = cv2.warpAffine(img, M[:2], (w, h))


show_image(warped, title="Scale(0.7) + Rotate(30 deg) around center", scale=5)

---
## 2. Homography

A **homography** (projective transform) maps points using a **general** $3 \times 3$ matrix:

$$H = \begin{pmatrix}h_{11}&h_{12}&h_{13}\\h_{21}&h_{22}&h_{23}\\h_{31}&h_{32}&1\end{pmatrix}$$

Unlike affine transforms (where the last row is always $(0, 0, 1)$), homography has non-zero $h_{31}, h_{32}$ — this creates **perspective distortion**. It has **8 DOF**, so we need **4 point correspondences** (each gives 2 equations) to solve for $H$.

A common application: given a photo of a planar object taken at an angle, warp it to a **frontal view**.

### Example: signboard rectification

**Steps:**
1. Identify 4 corner points of the planar object in the photo → **source points**
2. Define where those corners should map to (a rectangle) → **destination points**
3. `cv2.findHomography(src, dst)` → compute $H$
4. `cv2.warpPerspective(img, H, (w, h))` → warp the image

In [ ]:
img_sign = np.array(Image.open(IMG_DIR + "yangbanga.jpg"))

print(f"Shape: {img_sign.shape}")

# Show with axes visible so we can read pixel coordinates
fig, ax = plt.subplots(figsize=(5, 7))
ax.imshow(img_sign)
ax.set_title("yangbanga.jpg")
plt.tight_layout()
plt.show()

In [ ]:
# Step 1: define the 4 corners of the signboard (TL, TR, BR, BL)
src_pts = np.float32([[157, 286], [298, 221], [298, 356], [157, 402]])

# Visualize the quadrilateral on the image
fig, ax = plt.subplots(figsize=(5, 7))
ax.imshow(img_sign)
quad = plt.Polygon(src_pts, fill=False, edgecolor="lime", linewidth=2)
ax.add_patch(quad)
for i, (x, y) in enumerate(src_pts):
    ax.plot(x, y, "o", color="lime", markersize=8)
    ax.text(x + 8, y, f"P{i}", color="lime", fontsize=11, fontweight="bold")
ax.set_title("Source points on signboard")
plt.tight_layout()
plt.show()

In [ ]:
# Step 2: define destination rectangle + compute homography + warp
dst_pts = np.float32([[0, 0], [300, 0], [300, 200], [0, 200]])

H_sign, _ = cv2.findHomography(src_pts, dst_pts)
scanned = cv2.warpPerspective(img_sign, H_sign, (300, 200))

print(f"Homography matrix H:\n{np.round(H_sign, 4)}")
print(f"\nBottom row: h31={H_sign[2,0]:.6f}, h32={H_sign[2,1]:.6f}")

show_image(scanned, title="Rectified signboard", scale=5)

### Exercises

**Exercise 2.1:** Perform **document scanning** on the calligraphy hall sign (`img_hall`).

1. Examine the image below and estimate the 4 **source points** (corners of the sign plaque), in order: top-left, top-right, bottom-right, bottom-left.
2. Visualize your quadrilateral on the image (code provided) — adjust coordinates and **re-run the cell** until the green quad tightly fits the sign.
3. Compute the homography and warp to a **500 x 100** frontal rectangle (destination points provided below).

*Hint:* Read the $(x, y)$ coordinates from the axes. The sign is the dark rectangular plaque with gold characters — not the entire building facade.

In [ ]:
# Input — examine the axes to find corner coordinates
img_hall = np.array(Image.open(IMG_DIR + "calligraphy_hall.jpg"))

fig, ax = plt.subplots(figsize=(5, 7))
ax.imshow(img_hall)
ax.set_title("img_hall — find the 4 corners of the sign plaque")
plt.tight_layout()
plt.show()

# YOUR CODE HERE
# src_pts = np.float32([
#     [..., ...],   # top-left
#     [..., ...],   # top-right
#     [..., ...],   # bottom-right
#     [..., ...],   # bottom-left
# ])

# --- Visualize your quadrilateral (re-run until it fits) ---
fig, ax = plt.subplots(figsize=(5, 7))
ax.imshow(img_hall)
quad = plt.Polygon(src_pts, fill=False, edgecolor="lime", linewidth=2)
ax.add_patch(quad)
for i, (x, y) in enumerate(src_pts):
    ax.plot(x, y, "o", color="lime", markersize=8)
ax.set_title("Check: does the quad fit the sign?")
plt.tight_layout()
plt.show()

# --- Compute homography and warp ---
dst_pts = np.float32([[0, 0], [500, 0], [500, 100], [0, 100]])
H, _ = cv2.findHomography(src_pts, dst_pts)
scanned = cv2.warpPerspective(img_hall, H, (500, 100))

print(f"Homography matrix H:\n{np.round(H, 4)}")
show_image(scanned, title="Scanned sign (500 x 100)", scale=6)